In [15]:
NYC_BBOX   = "-74.2591,40.4774,-73.7004,40.9176"   
 
PARAMETERS = ["pm25", "no2", "o3"]
PAGE_LIMIT = 1000
MAX_PAGES  = 50
DATE_FROM  = "2024-01-01"
DATE_TO    = "2024-12-31"

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 17, Finished, Available, Finished, False)

In [16]:
import requests
import json
import os
import time
from datetime import datetime, date
 
api_key = notebookutils.credentials.getSecret(
    "https://fabric-project-kv.vault.azure.net", "openaq-api-key"
)
 
BRONZE_FILES = "/lakehouse/default/Files/air_quality"
os.makedirs(BRONZE_FILES, exist_ok=True)
 
BASE    = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": api_key, "Accept": "application/json"}

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 18, Finished, Available, Finished, False)

In [24]:
RETRYABLE = {408, 429, 500, 502, 503, 504}

def _get(url: str, params: dict = None, max_retries: int = 3) -> requests.Response:
    delay = 2.0
    for attempt in range(max_retries):
        r = requests.get(url, headers=HEADERS, params=params or {}, timeout=60)
        if r.status_code not in RETRYABLE:
            r.raise_for_status()
            return r
        wait = delay * (2 ** attempt)
        print(f"\n      [HTTP {r.status_code}] waiting {wait:.0f}s ...", end=" ", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"Exceeded {max_retries} retries on {url}")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 26, Finished, Available, Finished, False)

In [18]:
def get_parameter_ids(names: list[str]) -> dict[str, int]:
    r = _get(f"{BASE}/parameters")
    catalog = {p["name"].lower(): p["id"] for p in r.json().get("results", [])}
    resolved, missing = {}, []
    for name in names:
        pid = catalog.get(name.lower())
        if pid is None:
            missing.append(name)
        else:
            resolved[name] = pid
    if missing:
        raise ValueError(f"Parameters not found in OpenAQ catalog: {missing}")
    return resolved
 
 
print("Resolving parameter IDs ...")
param_ids = get_parameter_ids(PARAMETERS)
for name, pid in param_ids.items():
    print(f"  {name} -> {pid}")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 20, Finished, Available, Finished, False)

Resolving parameter IDs ...
  pm25 -> 2
  no2 -> 15
  o3 -> 32


In [19]:
def fetch_locations(bbox: str, pid_list: list[int]) -> list[dict]:
    locations, page = [], 1
    while True:
        r = _get(f"{BASE}/locations", {
            "bbox":          bbox,
            "parameters_id": ",".join(str(p) for p in pid_list),
            "limit":         PAGE_LIMIT,
            "page":          page,
        })
        data    = r.json()
        results = data.get("results", [])
        total   = int(str(data.get("meta", {}).get("found", 0)).lstrip(">"))
        print(f"  locations page {page}: {len(results)}  (total: {total})")
        if not results:
            break
        locations.extend(results)
        if len(locations) >= total or len(results) < PAGE_LIMIT:
            break
        page += 1
    return locations
 
 
print("\nDiscovering NYC locations ...")
locations = fetch_locations(NYC_BBOX, list(param_ids.values()))
print(f"  Found {len(locations)} locations")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 21, Finished, Available, Finished, False)


Discovering NYC locations ...
  locations page 1: 54  (total: 54)
  Found 54 locations


In [26]:
def fetch_location_sensors(location_id: int, target_param_names: set[str]) -> list[dict]:
    r = _get(f"{BASE}/locations/{location_id}/sensors")
    sensors = []
    for s in r.json().get("results", []):
        param = s.get("parameter", {})
        if param.get("name", "").lower() in target_param_names:
            sensors.append({
                "sensor_id":    s["id"],
                "location_id":  location_id,
                "parameter":    param.get("name", "").lower(),
                "parameter_id": param.get("id"),
                "unit":         param.get("units", ""),
            })
    return sensors
 
 
print("\nFetching sensors per location ...")
target_param_names = {p.lower() for p in PARAMETERS}   # {"pm25", "no2", "o3"}
target_sensors     = []

for i, loc in enumerate(locations, 1):
    lid      = loc["id"]
    loc_name = loc.get("name", str(lid))
    try:
        sensors = fetch_location_sensors(lid, target_param_names)
        for s in sensors:
            s["location_name"] = loc_name
        target_sensors.extend(sensors)
        print(f"  [{i:>3}/{len(locations)}] {loc_name[:35]:<35} {len(sensors)} sensors")
    except Exception as exc:
        print(f"  [{i:>3}/{len(locations)}] {loc_name[:35]:<35} skipped ({exc})")
    time.sleep(0.2)
 
by_param = {}
for s in target_sensors:
    by_param[s["parameter"]] = by_param.get(s["parameter"], 0) + 1
print(f"\n  Total sensors: {len(target_sensors)}")
for param, cnt in sorted(by_param.items()):
    print(f"    {param}: {cnt}")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 28, Finished, Available, Finished, False)


Fetching sensors per location ...
  [  1/54] CCNY                                2 sensors
  [  2/54] Manhattan/IS143                     1 sensors
  [  3/54] Bronx - IS52                        2 sensors
  [  4/54] Maspeth                             1 sensors
  [  5/54] Queens                              2 sensors
  [  6/54] PS 19                               1 sensors
  [  7/54] Bklyn - PS 314                      1 sensors
  [  8/54] Division Street                     1 sensors
  [  9/54] Bklyn - PS274                       1 sensors
  [ 10/54] Bronx - IS74                        1 sensors
  [ 11/54] Fort Lee Near Road                  2 sensors
  [ 12/54] Jersey City FH                      1 sensors
  [ 13/54] Elizabeth Trailer                   2 sensors
  [ 14/54] Newark Firehouse                    3 sensors
  [ 15/54] Queens Near-road                    1 sensors
  [ 16/54] Morrisania                          1 sensors
  [ 17/54] Union City High Scho                1 sens

In [27]:
from datetime import datetime, date, timedelta

def fetch_sensor_measurements(sensor_meta: dict) -> list[dict]:
    sid     = sensor_meta["sensor_id"]
    url     = f"{BASE}/sensors/{sid}/days"
    records = []
    page    = 1

    while page <= MAX_PAGES:
        r = _get(url, {
            "datetime_from": DATE_FROM,
            "datetime_to":   DATE_TO,
            "limit":         PAGE_LIMIT,
            "page":          page,
        })
        results = r.json().get("results", [])
        if not results:
            break

        for rec in results:
            rec["_sensor_id"]     = sid
            rec["_location_id"]   = sensor_meta["location_id"]
            rec["_location_name"] = sensor_meta["location_name"]
            rec["_parameter"]     = sensor_meta["parameter"]
            rec["_unit"]          = sensor_meta["unit"]

        records.extend(results)
        if len(results) < PAGE_LIMIT:
            break
        page += 1

    return records
 
 
print(f"\n{'='*60}")
print(f"  OpenAQ Bronze Ingestion")
print(f"  Range     : {DATE_FROM} -> {DATE_TO}")
print(f"  Sensors   : {len(target_sensors)}")
print(f"  Started   : {datetime.utcnow().isoformat()}Z")
print(f"{'='*60}\n")
 
all_measurements = []
skipped_sensors  = []
 
for idx, sensor in enumerate(target_sensors, 1):
    sid   = sensor["sensor_id"]
    label = f"[{idx:>3}/{len(target_sensors)}] sensor {sid:<12} ({sensor['parameter']})"
    print(f"  {label} ...", end=" ", flush=True)
 
    try:
        records = fetch_sensor_measurements(sensor)
        all_measurements.extend(records)
        print(f"{len(records):,} records")
    except requests.HTTPError as exc:
        code = exc.response.status_code
        print(f"HTTP {code} – skipped")
        skipped_sensors.append({"sensor_id": sid, "reason": f"HTTP {code}"})
    except Exception as exc:
        print(f"ERROR: {exc} – skipped")
        skipped_sensors.append({"sensor_id": sid, "reason": str(exc)})
 
    time.sleep(0.3)
 
print(f"\n  Records   : {len(all_measurements):,}")
print(f"  Succeeded : {len(target_sensors) - len(skipped_sensors)}/{len(target_sensors)} sensors")
print(f"  Skipped   : {len(skipped_sensors)} sensors (see skipped_sensors list)")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 29, Finished, Available, Finished, False)


  OpenAQ Bronze Ingestion
  Range     : 2024-01-01 -> 2024-12-31
  Sensors   : 62
  Started   : 2026-05-22T06:18:35.422735Z

  [  1/62] sensor 673          (pm25) ... 3,072 records
  [  2/62] sensor 671          (o3) ... 3,084 records
  [  3/62] sensor 1097         (pm25) ... 3,016 records
  [  4/62] sensor 1102         (pm25) ... 2,209 records
  [  5/62] sensor 1098         (o3) ... 3,193 records
  [  6/62] sensor 1103         (pm25) ... 3,096 records
  [  7/62] sensor 1152         (pm25) ... 2,754 records
  [  8/62] sensor 1106         (o3) ... 3,185 records
  [  9/62] sensor 1121         (pm25) ... 437 records
  [ 10/62] sensor 1128         (pm25) ... 3,215 records
  [ 11/62] sensor 1143         (pm25) ... 945 records
  [ 12/62] sensor 1145         (pm25) ... 2,671 records
  [ 13/62] sensor 1146         (pm25) ... 2,199 records
  [ 14/62] sensor 1534         (pm25) ... 3,385 records
  [ 15/62] sensor 1535         (no2) ... 3,505 records
  [ 16/62] sensor 5077566      (pm25) ... 1,0

In [28]:
run_date = date.today().isoformat()
out_file = os.path.join(BRONZE_FILES, f"openaq_{run_date}.json")
 
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(all_measurements, f, ensure_ascii=False, default=str)
 
size_mb = os.path.getsize(out_file) / 1_048_576
print(f"\n  [OK] Written {out_file}  ({size_mb:.2f} MB)")
 
if skipped_sensors:
    skip_file = os.path.join(BRONZE_FILES, f"openaq_skipped_{run_date}.json")
    with open(skip_file, "w") as f:
        json.dump(skipped_sensors, f, indent=2)
    print(f"  [OK] Skip log: {skip_file}")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 30, Finished, Available, Finished, False)


  [OK] Written /lakehouse/default/Files/air_quality/openaq_2026-05-22.json  (66.97 MB)


In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
 
spark = SparkSession.builder.getOrCreate()
 
raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(f"Files/air_quality/openaq_{run_date}.json")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(f"openaq_{run_date}.json"))
)
 
print("\nRaw schema:")
raw_df.printSchema()
print(f"Row count: {raw_df.count():,}")
 
(
    raw_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("air_quality_raw")
)
 
print("\n  [OK] Delta table 'air_quality_raw' updated.")

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 31, Finished, Available, Finished, False)


Raw schema:
root
 |-- _location_id: long (nullable = true)
 |-- _location_name: string (nullable = true)
 |-- _parameter: string (nullable = true)
 |-- _sensor_id: long (nullable = true)
 |-- _unit: string (nullable = true)
 |-- coordinates: string (nullable = true)
 |-- coverage: struct (nullable = true)
 |    |-- datetimeFrom: struct (nullable = true)
 |    |    |-- local: string (nullable = true)
 |    |    |-- utc: string (nullable = true)
 |    |-- datetimeTo: struct (nullable = true)
 |    |    |-- local: string (nullable = true)
 |    |    |-- utc: string (nullable = true)
 |    |-- expectedCount: long (nullable = true)
 |    |-- expectedInterval: string (nullable = true)
 |    |-- observedCount: long (nullable = true)
 |    |-- observedInterval: string (nullable = true)
 |    |-- percentComplete: double (nullable = true)
 |    |-- percentCoverage: double (nullable = true)
 |-- flagInfo: struct (nullable = true)
 |    |-- hasFlags: boolean (nullable = true)
 |-- parameter: stru

In [30]:
spark.sql("""
    SELECT
        _parameter                   AS parameter,
        COUNT(*)                     AS records,
        COUNT(DISTINCT _location_id) AS locations,
        ROUND(AVG(value), 4)         AS avg_value
    FROM air_quality_raw
    GROUP BY 1
    ORDER BY 1
""").show()

StatementMeta(, c40620c6-cd7d-4406-ad11-bb6b0f161867, 32, Finished, Available, Finished, False)

+---------+-------+---------+---------+
|parameter|records|locations|avg_value|
+---------+-------+---------+---------+
|      no2|   9260|        3|   0.0169|
|       o3|  11716|        4|   0.0255|
|     pm25|  47982|       54|    8.283|
+---------+-------+---------+---------+

